# AI Fix Verifier: demo

This notebook checks whether a security fix **removes a vulnerability without adding a new one**.
It scans the code before and after the fix with [Bandit](https://bandit.readthedocs.io/), compares the findings, and gives a verdict: **PASS**, **REJECT** or **REVIEW**.

**How to use:** click **Runtime → Run all**. Everything runs in about two minutes and needs no setup on your computer.

Repository: https://github.com/Sarsa-25/ai-fix-verifier

## 1. Setup
Download the project with all test-case branches, and install Bandit.

In [ ]:
!git clone -q https://github.com/Sarsa-25/ai-fix-verifier.git
%cd ai-fix-verifier
!git fetch -q --update-head-ok origin "+refs/heads/*:refs/heads/*"
!pip -q install bandit==1.9.4
!python -m bandit --version | head -1

## 2. One example: a fix that adds a new vulnerability

The fix below removes the SQL injection, but adds a hardcoded password. First, the code change:

In [ ]:
!git --no-pager diff main fix/sqli-with-secret -- vulnerable_app.py

Now the tool checks the fix. Expected: the SQL injection (B608) is **resolved**, the password (B105) is **introduced**, so the verdict is **REJECT**.

In [ ]:
!python verify.py main fix/sqli-with-secret vulnerable_app.py

## 3. The full test set: 19 cases

Each case is a pair of Git branches (before / after). The cases cover six scenario types: successful fix, incomplete fix, new vulnerability introduced, unrelated pre-existing finding, multiple findings, and ambiguous comparison.

"Expected" is a human judgement of each fix, set before running the tool.

In [ ]:
!python run_cases.py

In [ ]:
import glob
from IPython.display import Markdown, display

latest = sorted(glob.glob("outputs/*_summary.md"))[-1]
display(Markdown(open(latest, encoding="utf-8").read()))

## 4. Look at any case

Change `CASE` and run the cell to see the code change of that case (C05 to C19).

In [ ]:
CASE = "C19"   # try C05, C12, C17 (the three mismatches)

n = CASE.lower()
!git --no-pager diff case/{CASE}-before case/{CASE}-after -- cases/{n}.py

## Notes

- 16 of 19 verdicts match. The three mismatches are known limitations: C05 (Bandit flags every `subprocess` use, even a safe one), C12 (the strict policy counts an unrelated finding that was already there), C17 (deleting the vulnerable code looks like a fix to a static scanner).
- The fixes in this test set were written by hand with help from a chat assistant, not by an AI coding tool.
- More detail, limitations and two real-world checks are in the README.